# Topic Modeling
Unsupervised topic discovery on crisis video transcripts using BERTopic.

## Methodology

### Motivation
After scoring segments by sentiment, we apply topic modeling to uncover
*what* people are talking about — blame attribution, calls for justice,
grief, misinformation, political critique — giving qualitative structure
to the quantitative sentiment scores.

### Approach
We use **BERTopic** — an embedding-based topic model that:
1. Encodes each segment into a dense vector (sentence-transformers)
2. Reduces dimensionality (UMAP) for clustering stability
3. Clusters semantically similar segments (HDBSCAN)
4. Extracts descriptive terms per cluster (c-TF-IDF)

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1. Preprocessing** | Text cleaning | Lowercase, strip punctuation/URLs/digits, remove English + Filipino stopwords |
| **2. Embedding** | Sentence vectors | `paraphrase-multilingual-MiniLM-L12-v2` (12 languages, 384-d) |
| **3. Reduction** | UMAP | 384-d → 5-d for density-based clustering |
| **4. Clustering** | HDBSCAN | Soft clustering; non-assigned → noise cluster (-1) |
| **5. Representation** | c-TF-IDF | Top unigrams/bigrams per cluster as topic descriptors |
| **6. Labeling** | Manual annotation | Human-readable labels from top terms + sample segments |

### Cross-Analysis
Topic assignments are joined with sentiment labels, shouting flags,
and video IDs to answer:
- *Which topics dominate each news source?*
- *Do certain topics carry more negative sentiment?*
- *Are shouted segments concentrated in specific topics?*

### Input / Output
- **Input:** `scored_transcripts-{TRANSCRIPT_NAME}.csv` from sentiment analysis
- **Output:** Topic-labeled DataFrame + per-video topic profile + topic × sentiment cross-tab

In [1]:
import pandas as pd
import numpy as np
import ast
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import re
import string
import nltk

from nltk.corpus import stopwords
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import spacy

# Patch transformers for spacy-transformers compatibility
import transformers.tokenization_utils
import transformers.tokenization_utils_base
transformers.tokenization_utils.BatchEncoding = transformers.tokenization_utils_base.BatchEncoding

from dotenv import load_dotenv
from google import genai
from google.genai import types

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 150

## Configuration
Transcript selector. Change `TRANSCRIPT_NAME` to switch between
`ateneo_presscon, coach_tab`.

In [ ]:
# --- CONFIG ---
BASE_DIR = "../output"
TRANSCRIPT_NAME = "ateneo_presscon"  # or "coach_tab"
BATCH_PATH = os.path.join(BASE_DIR, f"scored_transcripts-{TRANSCRIPT_NAME}.csv")

## Load Batch
Read the scored transcript CSV. Each row is a cleaned transcript segment with
`source`, `start_time`, `end_time`, `sentiment_label`, `sentiment_score`, `is_shouting`, and
`sentences`. These features feed into cross-analysis
after topic modeling.

In [3]:
batch_df = pd.read_csv(BATCH_PATH)
print(f"Loaded {len(batch_df)} segments from scored_transcripts-{TRANSCRIPT_NAME}.csv")

# Parse the sentences column from stringified list back to Python list
batch_df['sentences'] = batch_df['sentences'].apply(ast.literal_eval)

batch_df[["text", "sentiment_label", "sentiment_score", "is_shouting"]].head()


Loaded 50 segments from scored_transcripts-coach_tab.csv


,text,sentiment_label,sentiment_score,is_shouting
0,"i'm coach tab baldwin, head coach of the atene...",NEUTRAL,0.769457,0
1,i'm here this morning to speak to the families...,POSITIVE,0.668303,0
2,and rene baterbonia.,NEUTRAL,0.461155,0
3,i've been asked to limit my comments this morn...,NEGATIVE,0.601227,0
4,in a situation in which i've been utterly lost...,NEGATIVE,0.795369,0


## Stopwords

Three sources merged into one set:

| Source | Provides | Origin |
|---|---|---|
| **nltk** | ~179 English stopwords | `nltk.corpus.stopwords` |
| **calamanCy** | ~147 Tagalog stopwords | spaCy defaults from `tl_calamancy_md` |
| **Custom** | ~30 noise words | `po`, `opo`, `kasi`, `lang`, `yung`, `nung`, etc. |

Custom additions cover common comment filler that isn't in
standard stopword lists.


In [4]:
nltk.download("stopwords", quiet=True)

en_stop = set(stopwords.words("english"))
nlp_tl = spacy.load("tl_calamancy_md")
tl_stop = nlp_tl.Defaults.stop_words

custom_stop = {
    "po", "opo", "kasi", "eh", "ah", "oh", "talaga", "naman",
    "lang", "pa", "na", "din", "rin", "raw", "daw", "yung",
    "nung", "sya", "ung", "ng", "sa", "ako", "ikaw", "tayo",
    "niya", "natin", "sila", "amin", "atin",
}

STOPWORDS = en_stop | tl_stop | custom_stop
print(f"Stopwords: {len(STOPWORDS)} total")
print(f"  English:  {len(en_stop)}")
print(f"  Tagalog:  {len(tl_stop)}")
print(f"  Custom:   {len(custom_stop)}")
print(f"Sample TL: {sorted(tl_stop)[:10]}")


Stopwords: 359 total
  English:  198
  Tagalog:  147
  Custom:   29
Sample TL: ['akin', 'aking', 'ako', 'alin', 'am', 'amin', 'aming', 'ang', 'ano', 'anumang']


## Text Cleaning

Apply a lightweight cleaning pipeline for topic modeling:

- Lowercase
- Remove URLs
- Strip punctuation
- Remove digits
- Filter stopwords + tokens shorter than 3 characters

Cleaned text is stored in a new `text_clean` column.
The original `text` column is preserved.


In [5]:
def clean_for_topics(text: str) -> str:
    text = text.lower()
    text = re.sub(r"https?://\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

batch_df["text"] = batch_df["text"].fillna("")

batch_df["text_clean"] = batch_df["text"].apply(clean_for_topics)
corpus = batch_df["text_clean"].tolist()

print(f"Corpus: {len(corpus)} documents")
print(f"Avg tokens/doc before: {np.mean([len(t.split()) for t in batch_df['text']]):.0f}")
print(f"Avg tokens/doc after:  {np.mean([len(t.split()) for t in corpus]):.0f}")


Corpus: 50 documents
Avg tokens/doc before: 14
Avg tokens/doc after:  7


## Embeddings

Encode each cleaned comment into a dense vector using
`sentence-transformers` for semantic clustering.

### Model

`paraphrase-multilingual-MiniLM-L12-v2` — chosen because:
- **Multilingual**: supports English + Tagalog + code-switched text
- **384-dimensional**: compact enough for UMAP reduction
- **Sentence-level**: trained on NLI/STS pairs for semantic similarity
  (critical for BERTopic's clustering quality)


In [6]:
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Encoding {len(corpus)} documents with {EMBEDDING_MODEL}...")
embeddings = embedding_model.encode(
    corpus,
    show_progress_bar=True,
    batch_size=64,
)
print(f"Embedding shape: {embeddings.shape}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 50 documents with paraphrase-multilingual-MiniLM-L12-v2...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (50, 384)


## BERTopic Model

Fit the topic model on the precomputed embeddings.

### Pipeline

| Step | Component | What it does |
|---|---|---|
| **1. Reduce** | UMAP (n_neighbors=15, n_components=5) | 384-d → 5-d for clustering stability |
| **2. Cluster** | HDBSCAN (min_cluster_size=10) | Density-based; unassigned → noise (-1) |
| **3. Reduce to 10** | BERTopic merge | Merges smaller topics into nearest larger topic until 10 remain (excl. noise) |
| **4. Represent** | c-TF-IDF | Top terms per cluster as topic descriptors |

### Why HDBSCAN over K-means?

| K-means | HDBSCAN (chosen) |
|---|---|
| Must set `k` upfront | Automatic — finds dense regions |
| Forces every point into a cluster | Noise cluster for off-topic segments |
| Spherical clusters only | Any shape (density-connected) |

For transcript segments, we don’t know the number of themes ahead of time.
HDBSCAN naturally finds them and isolates noise (spam, single-word segments).

### How HDBSCAN Works

Hierarchical Density-Based Spatial Clustering of Applications with Noise.

1. **Build a mutual reachability graph** — each segment (5-d UMAP point)
   is connected to its nearest neighbors. Dense regions get tight connections,
   sparse regions get loose ones.

2. **Create a hierarchy** — edges are removed in order of decreasing distance,
   producing a dendrogram of clusters at every density threshold.

3. **Extract stable clusters** — the algorithm selects clusters that persist
   across a wide range of density thresholds. Fluctuating clusters are pruned;
   only robust groupings remain.

4. **Assign noise** — points that never form a stable cluster are labeled `-1`.

### Why This Matters for Crisis Comments

| Property | Effect |
|---|---|
| **No fixed `k`** | HDBSCAN finds natural clusters; `nr_topics=10` merges smaller topics into the 10 most coherent |
| **Noise handling** | Spam, single words, or off-topic segments → `-1` instead of polluting a real topic |
| **Variable density** | Some themes are tight (e.g., “RIP” segments), others diffuse (e.g., “political critique”) — HDBSCAN handles both |
| **Outlier tolerance** | A heated segment with unusual vocabulary doesn’t force a new cluster or distort an existing one |

In [7]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=10,
    nr_topics=10,
    verbose=True,
)

topics, probs = topic_model.fit_transform(corpus, embeddings)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_noise = sum(1 for t in topics if t == -1)
print(f"\nTopics found: {n_topics}")
print(f"Noise cluster (-1): {n_noise} segments ({n_noise / len(topics):.1%})")


2026-07-04 00:38:49,726 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-04 00:39:05,527 - BERTopic - Dimensionality - Completed ✓
2026-07-04 00:39:05,529 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-04 00:39:05,588 - BERTopic - Cluster - Completed ✓
2026-07-04 00:39:05,589 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-07-04 00:39:05,675 - BERTopic - Representation - Completed ✓
2026-07-04 00:39:05,676 - BERTopic - Topic reduction - Reducing number of topics
2026-07-04 00:39:05,676 - BERTopic - Topic reduction - Number of topics (10) is equal or higher than the clustered topics(3).
2026-07-04 00:39:05,679 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-04 00:39:06,187 - BERTopic - Representation - Completed ✓



Topics found: 2
Noise cluster (-1): 11 segments (22.0%)


In [8]:
topic_model.get_topic_info().head(15)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,11,-1_families_people_two_adili,"[families, people, two, adili, grief, loss, ca...",[morning speak families two basketball players...
1,0,13,0_coach_failed_basketball_players,"[coach, failed, basketball, players, felt, hel...",[coach tab baldwin head coach ateneo blue eagl...
2,1,26,1_rene_pray_ive_sorry,"[rene, pray, ive, sorry, divine, everybody, ne...",[beseech everybody cared loves love rene divin...


In [9]:
topic_model.visualize_barchart(top_n_topics=10, n_words=10)

In [10]:
topic_model.visualize_distribution(probs, min_probability=0.05)

## Topic Labeling

Each topic cluster is labeled by `gemma-4-31b-it` via the Gemini API.
Top 10 c-TF-IDF terms + 5 sample segments are sent per topic.
Labels are reviewed and manually overridden as needed.


In [27]:
load_dotenv("../../.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# MODEL = "gemma-4-31b-it"
MODEL = "gemini-2.5-flash"


LLM_LABELS = {}
for topic_id in sorted([t for t in set(topics) if t != -1]):
    terms = topic_model.get_topic(topic_id)
    term_str = ", ".join([t[0] for t in terms[:10]])

    mask = [t == topic_id for t in topics]
    samples = batch_df.loc[mask, "text"].sample(min(5, sum(mask))).tolist()

    prompt = (
        "You are a crisis discourse analyst. Given the top terms "
        "and sample segments for a topic cluster, assign a short "
        "label (2-4 words).\n\n"
        f"Top terms: {term_str}\n"
        "Sample segments:\n"
        + "\n".join(f"- {s[:200]}" for s in samples[:5])
        + "\n\nLabel (2-4 words):"
    )

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0),
    )
    label = response.text.strip()
    LLM_LABELS[topic_id] = label
    print(f"Topic {topic_id}: {label}")


Topic 0: Coach's Failed Trust
Topic 1: Loss, Remorse, Solace


In [28]:
# Manual label overrides — fix typos or rename topics
# Uncomment and edit lines below, then re-run from here:
LABEL_OVERRIDES = {
    # 3: "Coach Accountability",
    # 5: "Demand for Justice (corrected)",
}

LLM_LABELS.update(LABEL_OVERRIDES)
FINAL_LABELS = LLM_LABELS.copy()

for tid, label in sorted(FINAL_LABELS.items()):
    print(f"  Topic {tid}: {label}")


  Topic 0: Coach's Failed Trust
  Topic 1: Loss, Remorse, Solace


In [29]:
batch_df["topic_id"] = topics
batch_df["topic_label"] = batch_df["topic_id"].map(FINAL_LABELS)
batch_df[["text_clean", "topic_id", "topic_label"]].head(10)


,text_clean,topic_id,topic_label
0,coach tab baldwin head coach ateneo blue eagles,0,Coach's Failed Trust
1,morning speak families two basketball players ...,-1,NaN
2,rene baterbonia,1,"Loss, Remorse, Solace"
3,ive asked limit comments morning four minutes,1,"Loss, Remorse, Solace"
4,situation ive utterly lost words four minutes ...,1,"Loss, Remorse, Solace"
5,eternity yet opportunity speak rene divine bel...,1,"Loss, Remorse, Solace"
6,could speak hours gone left behind,1,"Loss, Remorse, Solace"
7,left behind carrying immense sorrow carrying i...,-1,NaN
8,carrying immense remorse ive known since momen...,1,"Loss, Remorse, Solace"
9,passed away matter magnitude grief two familie...,-1,NaN


## Cross-Analysis

Topic labels are joined with sentiment, shouting, to answer the research questions from the methodology:

- *Do certain topics carry more negative sentiment?*
  → Topic × sentiment_label heatmap

- *Are shouted segments concentrated in specific topics?*
  → Shouting ratio per topic


In [30]:
topic_x_sentiment = pd.crosstab(
    batch_df["topic_label"],
    batch_df["sentiment_label"],
    margins=True,
    margins_name="Total",
)
topic_x_sentiment


sentiment_label,NEGATIVE,NEUTRAL,POSITIVE,Total
topic_label,,,,
Coach's Failed Trust,4,5,4,13
"Loss, Remorse, Solace",15,4,7,26
Total,19,9,11,39


In [31]:
shouting_by_topic = batch_df.groupby("topic_label")["is_shouting"].agg(
    total="count",
    shouted="sum",
)
shouting_by_topic["shouting_ratio"] = shouting_by_topic["shouted"] / shouting_by_topic["total"]
shouting_by_topic.sort_values("shouting_ratio", ascending=False)


,total,shouted,shouting_ratio
topic_label,,,
Coach's Failed Trust,13,0,0.0
"Loss, Remorse, Solace",26,0,0.0


## Export

Save the topic-labeled segments for downstream use or
comparison across batches.

Columns preserved: text, source, start_time, end_time, sentiment_label, sentiment_score,
is_shouting, topic_id, topic_label, sentences.


In [32]:
output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"topics_transcripts-{TRANSCRIPT_NAME}.csv")
batch_df.to_csv(output_path, index=False)
print(f"Exported {len(batch_df)} topic-labeled segments to {output_path}")


Exported 50 topic-labeled segments to ../output/topics_transcripts-coach_tab.csv


In [33]:
# Post-hoc topic rename — edit labels in the exported CSV
# Run this cell after export to fix typos without re-running the pipeline.
df = pd.read_csv(output_path)

current = df[["topic_id", "topic_label"]].drop_duplicates().sort_values("topic_id")
display(current)

LABEL_RENAMES = {
    # 0: "Drowning Accountability (corrected)",
}

for tid, label in LABEL_RENAMES.items():
    df.loc[df["topic_id"] == tid, "topic_label"] = label

df.to_csv(output_path, index=False)
print("Labels updated in CSV.")


,topic_id,topic_label
1,-1,NaN
0,0,Coach's Failed Trust
2,1,"Loss, Remorse, Solace"


Labels updated in CSV.
